# Internal Benchmarking — All Methods, Both Years

Evaluates all 10 prediction methods on the **sampled test set** (5:1 negative ratio, inflated positive rate) for both **t=2015** and **t=2016**.

Combines and supersedes `evaluation.ipynb` + `filtered_metrics_eval.ipynb`.

### What this notebook does
1. **t=2015 full sampled test** (127,531 pairs, ~14.5% positive) — all 10 methods, 8 metrics
2. **t=2015 RCA>0.25 filtered** — same 10 methods, 8 metrics on near-miss subset
3. **t=2016 full sampled test** (generated fresh at 5:1 ratio) — all 10 methods, 8 metrics
4. **t=2016 RCA>0.25 filtered** — same 10 methods, 8 metrics on near-miss subset

### Metrics
| Metric | Definition |
|--------|------------|
| PR-AUC | Area under Precision-Recall curve (primary metric for imbalanced data) |
| AUROC | Area under ROC curve |
| NDCG@20 | Normalised DCG@20, macro-averaged per country |
| Prec@20 | Precision@20, macro-averaged per country |
| CWR | Complexity-Weighted Recall: top-50% percentile recall, weighted by 1/ubiquity (fixed: fillna(median)) |
| Best F1 | F1 at the threshold maximising F1 |
| P@1000 | Precision@1000 globally |
| mAP@10 | Mean Average Precision@10, per country |

**CWR fix applied:** products missing from 2010 ubiquity reference now use `fillna(ubiq_median/max_ubiq)` instead of 0.0 — prevents absent products from receiving the maximum rarity weight.

### Methods
1. RCA Persistence  
2. Density (Product Space)  
3. ECI  
4. ECI + Density  
5. KNN (LLM embeddings)  
6. GNN-4F (BACI only)  
7. GNN-11F (BACI+WDI)  
8. GNN-11F+LLM (SAGEConv + capability edges)  
9. GNN-LLM v2 (GAT + Focal Loss + Optuna)  
10. GNN-LLM v2 Unopt (GAT + Focal Loss, fixed hparams, val PR-AUC=0.3211)

## Setup
Load all artifacts, define metric helpers, build shared structures.

In [15]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score, ndcg_score
from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv, GATConv, GCNConv, to_hetero
warnings.filterwarnings('ignore')

DATA_DIR     = 'data'
TRAIN_CUTOFF = 2012
NEG_RATIO    = 5
LABEL_HORIZON = 5
RCA_THRESH   = 0.25
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR     = os.path.join(DATA_DIR, 'models', 'gnn', 'checkpoints')
PCA_DIM      = 32
print(f'Device: {DEVICE}')

# ── Raw data ──────────────────────────────────────────────────────────────────
smooth    = pd.read_csv(os.path.join(DATA_DIR, 'M_cpt_smoothed.csv'))
rca_df    = pd.read_csv(os.path.join(DATA_DIR, 'rca_cpt.csv'))
test_lbl_2015 = pd.read_csv(os.path.join(DATA_DIR, 'test_labels.csv'))
train_lbl = pd.read_csv(os.path.join(DATA_DIR, 'train_labels.csv'))
val_lbl   = pd.read_csv(os.path.join(DATA_DIR, 'val_labels.csv'))

countries = sorted(smooth['country'].unique())
products  = sorted(smooth['product'].unique())
C, P = len(countries), len(products)
c_idx = {c: i for i, c in enumerate(countries)}
p_idx = {p: i for i, p in enumerate(products)}

def build_M(year):
    M = np.zeros((C, P), dtype=np.float32)
    yr = smooth[smooth['year'] == year]
    M[yr['country'].map(c_idx).values, yr['product'].map(p_idx).values] = 1.0
    return M

# ── Proximity matrix (train years only — no leakage) ─────────────────────────
print('Building proximity matrix from training years...')
co_exp  = np.zeros((P, P), dtype=np.float32)
any_exp = np.zeros((P, P), dtype=np.float32)
for yr in sorted(y for y in smooth['year'].unique() if y <= TRAIN_CUTOFF):
    M = build_M(yr)
    co = M.T @ M
    ex = M.sum(axis=0)
    co_exp  += co
    any_exp += ex[:, None] + ex[None, :] - co
phi = np.where(any_exp > 0, co_exp / (any_exp + 1e-9), 0.0)
np.fill_diagonal(phi, 0.0)
phi_row_sum = phi.sum(axis=1)

# ── PCI weights for CWR (fixed: fillna median, not 0) ────────────────────────
rca_ref  = rca_df[rca_df['year'] == 2010]
ubiq     = rca_ref.groupby('product')['rca'].apply(lambda x: (x >= 1).sum())
max_ubiq = ubiq.max()
ubiq_median = ubiq.median()
pci_dict = {int(p): float(-u / max_ubiq) for p, u in ubiq.items()}
pci_fill = float(-ubiq_median / max_ubiq)   # median-based fill (not 0)

def add_pci_weights(df):
    """Add 'pci' and 'w' columns to a label dataframe."""
    df = df.copy()
    df['pci'] = df['product'].map(pci_dict).fillna(pci_fill)
    min_pci   = df['pci'].min()
    df['w']   = df['pci'] - min_pci
    return df

def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)

# ── GNN tensor artifacts ──────────────────────────────────────────────────────
edge_idx_raw   = torch.load(os.path.join(DATA_DIR, 'edge_index_by_year.pt'), weights_only=False)
edge_idx_by_yr = {k: v.long() for k, v in edge_idx_raw.items()}
p_x_by_yr      = torch.load(os.path.join(DATA_DIR, 'product_x_by_year.pt'), weights_only=False)
c_x_11feat     = torch.load(os.path.join(DATA_DIR, 'country_x_by_year.pt'), weights_only=False)
cap_ei         = torch.load(os.path.join(DATA_DIR, 'capability_edge_index.pt'), weights_only=False).long()

with open(os.path.join(DATA_DIR, 'country_mapping.pkl'), 'rb') as f: c_map = pickle.load(f)
with open(os.path.join(DATA_DIR, 'product_mapping.pkl'), 'rb') as f: p_map = pickle.load(f)

c_feat_df = pd.read_csv(os.path.join(DATA_DIR, 'country_features.csv'))
BACI_COLS = ['log_export', 'n_products', 'avg_rca', 'max_rca']
c_x_4feat = {}
for yr in sorted(c_feat_df['year'].unique()):
    yd = c_feat_df[c_feat_df['year'] == yr].copy()
    yd['idx'] = yd['country'].map(c_map['to_idx'])
    yd = yd.dropna(subset=['idx']).sort_values('idx')
    c_x_4feat[int(yr)] = torch.tensor(yd[BACI_COLS].values, dtype=torch.float32)

# ── LLM embeddings for KNN + GNN-v2 ──────────────────────────────────────────
llm_emb_t = torch.load(os.path.join(DATA_DIR, 'product_llm_embeddings.pt'),
                        weights_only=False, map_location='cpu').float()
llm_emb   = llm_emb_t.numpy()   # [P, 768], unit-normed

# Augmented product features for v2: [P, 771]
p_x_by_yr_v2 = {yr: torch.cat([base, llm_emb_t], dim=1)
                for yr, base in p_x_by_yr.items()}

# Cosine edge weights for capability edges: [144192, 1]
emb_np = llm_emb
src_np = cap_ei[0].numpy(); dst_np = cap_ei[1].numpy()
cos_weights = torch.tensor(
    (emb_np[src_np] * emb_np[dst_np]).sum(axis=1), dtype=torch.float32
).unsqueeze(1)

# Scalar cosine weights (no unsqueeze) for GCNConv in Variant B
cos_weights_1d = cos_weights.squeeze(1)   # [144192]

P_IN_V2 = 771
C_IN    = 11

# ── PCA-compressed LLM features (for Variants A & B) ─────────────────────────
pca = PCA(n_components=PCA_DIM, random_state=42)
pca.fit(llm_emb)
explained = pca.explained_variance_ratio_.sum()
llm_pca_np = pca.transform(llm_emb).astype(np.float32)
# L2-normalise rows so scale matches BACI features
llm_pca_np /= np.linalg.norm(llm_pca_np, axis=1, keepdims=True).clip(min=1e-8)
llm_pca_t   = torch.from_numpy(llm_pca_np)   # [5018, 32]

P_IN_PCA = 3 + PCA_DIM   # 35
p_x_with_pca = {yr: torch.cat([base, llm_pca_t], dim=1)
                for yr, base in p_x_by_yr.items()}

print(f'Countries: {C}  Products: {P}')
print(f'Proximity matrix built from {TRAIN_CUTOFF} and earlier.')
print(f'LLM embeddings: {llm_emb.shape}  |  cos_weights: {cos_weights.shape}')
print(f'PCA({PCA_DIM}d) explains {explained*100:.1f}% variance  |  P_IN_PCA={P_IN_PCA}')
print(f'GNN checkpoints dir: {CKPT_DIR}')

Device: cuda
Building proximity matrix from training years...
Countries: 233  Products: 5018
Proximity matrix built from 2012 and earlier.
LLM embeddings: (5018, 768)  |  cos_weights: torch.Size([144192, 1])
PCA(32d) explains 61.6% variance  |  P_IN_PCA=35
GNN checkpoints dir: data\models\gnn\checkpoints


## Label Builder

For **t=2015**: loads `test_labels.csv` directly (already generated by pipeline Step 4).

For **t=2016**: generates fresh labels using the same Step 4 logic:
- Positives: `M[2016]=0` AND `M[2021]=1` AND `M[2022]=1` (sustained new transition, h=5)
- Negatives: `M[2016]=0` AND `M[2021]=0`  
- Sampling: 5 negatives per positive (seed=42)
- Year column set to 2016 for GNN snapshot alignment

In [16]:
def build_sampled_labels(t, neg_ratio=NEG_RATIO, seed=42):
    """Return a label dataframe for observation year t at 5:1 neg ratio."""
    if t == 2015:
        df = test_lbl_2015.copy()
        if 'year' not in df.columns:
            df['year'] = 2015
        return df

    rng = np.random.default_rng(seed)
    M_t      = build_M(t)
    M_tp5    = build_M(t + LABEL_HORIZON)       # M[2021]
    M_tp6    = build_M(t + LABEL_HORIZON + 1)   # M[2022]

    ci_arr_all = np.repeat(np.arange(C), P)
    pi_arr_all = np.tile(np.arange(P), C)

    absent = M_t[ci_arr_all, pi_arr_all] == 0
    pos    = absent & (M_tp5[ci_arr_all, pi_arr_all] == 1) & (M_tp6[ci_arr_all, pi_arr_all] == 1)
    neg    = absent & (M_tp5[ci_arr_all, pi_arr_all] == 0)

    pos_idx = np.where(pos)[0]
    neg_idx = np.where(neg)[0]

    n_neg = min(len(neg_idx), len(pos_idx) * neg_ratio)
    neg_sampled = rng.choice(neg_idx, size=n_neg, replace=False)

    idx_all = np.concatenate([pos_idx, neg_sampled])
    labels  = np.concatenate([np.ones(len(pos_idx)), np.zeros(n_neg)])

    # Map back to country/product names
    idx_to_country = {i: c for c, i in c_idx.items()}
    idx_to_product = {i: p for p, i in p_idx.items()}

    df = pd.DataFrame({
        'country': [idx_to_country[ci_arr_all[i]] for i in idx_all],
        'product': [idx_to_product[pi_arr_all[i]] for i in idx_all],
        'label':   labels.astype(int),
        'year':    t,
    })
    return df.reset_index(drop=True)


labels_2015 = build_sampled_labels(2015)
labels_2016 = build_sampled_labels(2016)

for yr, df in [(2015, labels_2015), (2016, labels_2016)]:
    n, pos = len(df), df['label'].sum()
    print(f't={yr}: {n:,} pairs  |  {pos:,} positives  |  {pos/n*100:.1f}% positive rate')

t=2015: 127,531 pairs  |  18,477 positives  |  14.5% positive rate
t=2016: 112,284 pairs  |  18,714 positives  |  16.7% positive rate


## Metric Functions

All 8 metrics computed on a single call. Scores and labels must be aligned 1:1 with the label dataframe passed in.

In [17]:
def compute_metrics(scores, df_lbl):
    """
    Compute all 8 metrics on a label dataframe.
    scores: 1-D array aligned with df_lbl rows.
    df_lbl: must have columns 'label', 'country', 'product', 'w' (PCI weight).
    """
    scores = np.asarray(scores, dtype=np.float64)
    labels = df_lbl['label'].values

    # PR-AUC
    prec, rec, _ = precision_recall_curve(labels, scores)
    pr_auc = auc(rec, prec)

    # AUROC
    auroc = roc_auc_score(labels, scores) if 0 < labels.mean() < 1 else 0.0

    # Best F1
    denom  = prec + rec
    f1_all = np.where(denom > 0, 2 * prec * rec / denom, 0.0)
    best_f1 = float(f1_all.max())

    # P@1000
    K = min(1000, len(scores))
    topk = np.argsort(scores)[::-1][:K]
    p1k  = float(labels[topk].sum()) / K

    # NDCG@20, Prec@20, mAP@10 — per country
    df = df_lbl.copy()
    df['score'] = scores
    ndcg_vals, prec20_vals, ap10_vals = [], [], []
    for _, grp in df.groupby('country'):
        if grp['label'].sum() == 0:
            continue
        yt = grp['label'].values
        ys = grp['score'].values
        try:
            ndcg_vals.append(ndcg_score([yt], [ys], k=20))
        except Exception:
            pass
        prec20_vals.append(grp.sort_values('score', ascending=False).head(20)['label'].mean())
        n_pos = int(grp['label'].sum())
        top10 = grp.sort_values('score', ascending=False).head(10)['label'].values
        cumtp = np.cumsum(top10)
        prec_k = cumtp / np.arange(1, len(top10) + 1)
        ap10_vals.append((prec_k * top10).sum() / min(n_pos, 10))

    ndcg20 = float(np.nanmean(ndcg_vals))   if ndcg_vals   else 0.0
    prec20 = float(np.nanmean(prec20_vals)) if prec20_vals else 0.0
    map10  = float(np.nanmean(ap10_vals))   if ap10_vals   else 0.0

    # CWR (fixed: percentile rank, median fill for missing products)
    df['score_pct'] = df['score'].rank(pct=True)
    tot_w = df.loc[df['label'] == 1, 'w'].sum()
    hit_w = df.loc[(df['label'] == 1) & (df['score_pct'] >= 0.5), 'w'].sum()
    cwr   = float(hit_w / tot_w) if tot_w > 0 else 0.0

    return {
        'PR-AUC':  round(pr_auc, 4),
        'AUROC':   round(auroc,  4),
        'NDCG@20': round(ndcg20, 4),
        'Prec@20': round(prec20, 4),
        'CWR':     round(cwr,    4),
        'Best F1': round(best_f1, 4),
        'P@1000':  round(p1k,    4),
        'mAP@10':  round(map10,  4),
    }


# ── Results store ─────────────────────────────────────────────────────────────
# RESULTS[year][name]        -> metrics dict (full sampled set)
# RESULTS_FILT[year][name]   -> metrics dict (RCA>0.25 filtered set)
# SCORES[year][name]         -> raw score array aligned with labels_YEAR
RESULTS      = {2015: {}, 2016: {}}
RESULTS_FILT = {2015: {}, 2016: {}}
SCORES       = {2015: {}, 2016: {}}
LABELS       = {2015: add_pci_weights(labels_2015), 2016: add_pci_weights(labels_2016)}
INVALID_MAP10 = set()   # methods where mAP@10 is meaningless (ECI)

def evaluate(t, name, scores, skip_map10=False):
    """Score a method for year t. Caches scores for filtered evaluation later."""
    scores = np.asarray(scores, dtype=np.float64)
    SCORES[t][name] = scores
    df = LABELS[t]
    res = compute_metrics(scores, df)
    if skip_map10:
        res['mAP@10'] = 'N/A'
        INVALID_MAP10.add(name)
    RESULTS[t][name] = res
    tag = 'N/A' if res['mAP@10'] == 'N/A' else f'{res["mAP@10"]:.4f}'
    print(f'  [{t}] {name:<30}  PR-AUC={res["PR-AUC"]:.4f}  '
          f'NDCG@20={res["NDCG@20"]:.4f}  Prec@20={res["Prec@20"]:.4f}  '
          f'CWR={res["CWR"]:.4f}  P@1000={res["P@1000"]:.4f}  mAP@10={tag}')
    return res

print('Metric functions ready.')
print(f'PCI fill for missing products: {pci_fill:.4f}  (median-ubiquity, not 0)')

Metric functions ready.
PCI fill for missing products: -0.1935  (median-ubiquity, not 0)


## ECI Builder (per year)

Builds ECI, density, and M matrices for a given observation year.

In [18]:
def build_year_structures(t):
    """Returns (M_t, dens_mat, eci) for observation year t."""
    M_t = build_M(t)
    kc  = M_t.sum(axis=1); kp = M_t.sum(axis=0)
    kc_s = np.where(kc > 0, kc, 1.0); kp_s = np.where(kp > 0, kp, 1.0)
    kc_n, kp_n = kc.astype(float), kp.astype(float)
    for _ in range(20):
        kc_n = (1.0 / kc_s) * (M_t   @ kp_n)
        kp_n = (1.0 / kp_s) * (M_t.T @ kc_n)
    eci = (kc_n - kc_n.mean()) / (kc_n.std() + 1e-9)
    dens_mat = (M_t @ phi) / (phi_row_sum[None, :] + 1e-9)
    return M_t, dens_mat, eci

print('Year-structure builder ready.')

Year-structure builder ready.


## GNN Architecture & Helpers

Defines all model classes (SAGEConv v1, GATConv v2) and inference helpers for the sampled test set.

## XGBoost — Load & Inference

Loads the pre-trained XGBoost checkpoint (trained via `ClaudeFiles/train_xgboost.py`).
Features: 11 country (BACI+WDI) + 3 product + 32 PCA-LLM + density + ECI + 3×RCA-history = 51 dims.

In [ ]:
import xgboost as xgb
import pickle as pkl

XGB_CKPT = os.path.join(DATA_DIR, 'models', 'xgboost', 'xgb_model.pkl')
if not os.path.exists(XGB_CKPT):
    raise FileNotFoundError(
        f'XGBoost checkpoint not found at {XGB_CKPT}. '
        'Run ClaudeFiles/train_xgboost.py first.')

with open(XGB_CKPT, 'rb') as _f:
    _xgb_bundle = pkl.load(_f)
xgb_model = _xgb_bundle['model']
xgb_pca   = _xgb_bundle['pca']

# Apply PCA once — reuse for every inference call
xgb_llm_pca = xgb_pca.transform(llm_emb).astype('float32')
xgb_llm_pca /= np.linalg.norm(xgb_llm_pca, axis=1, keepdims=True).clip(min=1e-8)

# Pre-build country and product lookup tables indexed by (ci, pi)
_c_enrich = pd.read_csv(os.path.join(DATA_DIR, 'country_features_enriched.csv'))
_p_feat   = pd.read_csv(os.path.join(DATA_DIR, 'product_features.csv'))
_CCOLS = ['log_export', 'n_products', 'avg_rca', 'max_rca',
          'gdp_pc', 'capital_formation', 'tertiary_enrollment',
          'fdi_inflows', 'manufacturing_va', 'internet_users', 'population']
_PCOLS = ['log_world_export', 'ubiquity', 'avg_rca']

def _build_lookup(feat_df, id_col, idx_map, all_ids, cols, n_ids):
    """Build a dense [n_ids, len(cols)] array for fast index-based lookup."""
    arr = np.zeros((n_ids, len(cols)), dtype=np.float32)
    avail_yrs = sorted(feat_df['year'].unique())
    # Use max train year (2012) for lookups to avoid leakage
    yr = max(y for y in avail_yrs if y <= TRAIN_CUTOFF)
    sub = feat_df[feat_df['year'] == yr].set_index(id_col)[cols]
    for eid, ei in idx_map.items():
        if eid in sub.index:
            arr[ei] = sub.loc[eid].values.astype(np.float32)
    return arr

_c_arr = _build_lookup(_c_enrich, 'country', c_idx, countries, _CCOLS, C)  # [C, 11]
_p_arr = _build_lookup(_p_feat,   'product', p_idx, products,  _PCOLS, P)  # [P,  3]

# Pre-build dense RCA history array [C, P, 3] for all train+test years
# We rebuild per-call for the correct t, but the rca_df filter is vectorized.
print(f'XGBoost loaded  |  PCA-LLM: {xgb_llm_pca.shape}  |  country arr: {_c_arr.shape}')

try:
    import cupy as cp
    _USE_CUPY = True
    print('cupy available — GPU prediction enabled')
except ImportError:
    _USE_CUPY = False
    print('cupy not available — using CPU prediction (still fast)')

def build_xgb_features(df_lbl, t):
    """Fully vectorized 51-dim feature matrix. No Python loops."""
    ci = df_lbl['country'].map(c_idx).values.astype(np.int64)
    pi = df_lbl['product'].map(p_idx).values.astype(np.int64)

    # ECI and density (GPU)
    M_t, dens_t, eci_t = build_year_structures(t)
    dens_batch = dens_t[ci, pi].reshape(-1, 1)
    eci_batch  = eci_t[ci].reshape(-1, 1)

    # Country + product + LLM (array index — O(1) per feature block)
    c_batch   = _c_arr[ci]          # [N, 11]
    p_batch   = _p_arr[pi]          # [N,  3]
    llm_batch = xgb_llm_pca[pi]     # [N, 32]

    # RCA history — build dense [C, P, 3] and index
    hist_yrs = [t - 2, t - 1, t]
    rca_sub  = rca_df[rca_df['year'].isin(hist_yrs)].copy()
    rca_sub['ci2'] = rca_sub['country'].map(c_idx)
    rca_sub['pi2'] = rca_sub['product'].map(p_idx)
    rca_sub  = rca_sub.dropna(subset=['ci2', 'pi2'])
    rca_sub['ci2'] = rca_sub['ci2'].astype(int)
    rca_sub['pi2'] = rca_sub['pi2'].astype(int)
    rca_arr  = np.zeros((C, P, 3), dtype=np.float32)
    for k, yr in enumerate(hist_yrs):
        rows = rca_sub[rca_sub['year'] == yr]
        rca_arr[rows['ci2'].values, rows['pi2'].values, k] = rows['rca'].values.astype(np.float32)
    rca_batch = rca_arr[ci, pi]     # [N, 3]

    return np.hstack([c_batch, p_batch, llm_batch, dens_batch, eci_batch, rca_batch])

def xgb_scores_sampled(t, df_lbl):
    """XGBoost probability scores — GPU prediction via inplace_predict."""
    X = build_xgb_features(df_lbl, t).astype(np.float32)
    if _USE_CUPY:
        X_gpu = cp.asarray(X)
        return np.asarray(xgb_model.inplace_predict(X_gpu)).astype(np.float64)
    return xgb_model.inplace_predict(X).astype(np.float64)

print('XGBoost inference ready (vectorized + GPU predict).')


In [20]:
# ── SAGEConv architecture (Methods 6, 7, 8) ───────────────────────────────────
class _HomoGNN(nn.Module):
    def __init__(self, hidden, drop=0.3):
        super().__init__()
        self.c1 = SAGEConv(hidden, hidden)
        self.c2 = SAGEConv(hidden, hidden)
        self.drop = drop
    def forward(self, x, edge_index):
        x = F.dropout(self.c1(x, edge_index).relu(), p=self.drop, training=self.training)
        return self.c2(x, edge_index)

class BipartiteEncoder(nn.Module):
    def __init__(self, c_in, hidden, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(3, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden), meta)
    def forward(self, x_dict, ei_dict):
        return self.gnn({'country': self.country_lin(x_dict['country']),
                         'product': self.product_lin(x_dict['product'])}, ei_dict)

class TemporalGNN(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

# ── GATConv architecture (Methods 9 & 10 — v2 variants) ──────────────────────
class _GATBlock(nn.Module):
    def __init__(self, hidden, heads, drop):
        super().__init__()
        self.gat1 = GATConv(hidden, hidden, heads=heads, concat=True,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.proj = nn.Linear(hidden * heads, hidden)
        self.gat2 = GATConv(hidden, hidden, heads=1, concat=False,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.drop = drop
    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)

class BipartiteEncoderGAT(nn.Module):
    def __init__(self, c_in, p_in, hidden, heads, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock(hidden, heads, drop), meta)
    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {'country': self.country_lin(x_dict['country']),
                  'product': self.product_lin(x_dict['product'])}
        return self.gnn(x_proj, ei_dict, ea_dict) if ea_dict else self.gnn(x_proj, ei_dict)

class TemporalGNNv2(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ea in snaps:   # snaps is list of (HeteroData, ea_dict|None)
            z = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

# ── Variant A: SAGEConv + PCA features (Methods 11) ──────────────────────────
class BipartiteEncoderSAGE_PCA(nn.Module):
    def __init__(self, c_in, p_in, hidden, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_HomoGNN(hidden, drop), meta)
    def forward(self, snap, _ew=None):
        x_proj = {'country': self.country_lin(snap['country'].x),
                  'product': self.product_lin(snap['product'].x)}
        return self.gnn(x_proj, snap.edge_index_dict)

class TemporalGNN_PCA(nn.Module):
    """Accepts list of (HeteroData, ew|None) tuples."""
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ew in snaps:
            z = self.enc(s, ew)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}

# ── Variant B: Mixed SAGEConv + GCNConv with cosine edge weights (Method 12) ──
class MixedBipartiteEncoder(nn.Module):
    def __init__(self, c_in, p_in, hidden, drop):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.drop = drop
        self.sage_exp_1  = SAGEConv(hidden, hidden)
        self.sage_rexp_1 = SAGEConv(hidden, hidden)
        self.gcn_cap_1   = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)
        self.sage_exp_2  = SAGEConv(hidden, hidden)
        self.sage_rexp_2 = SAGEConv(hidden, hidden)
        self.gcn_cap_2   = GCNConv(hidden, hidden, add_self_loops=False, normalize=False)

    def _layer(self, xc, xp, snap, cap_ew, sage_exp, sage_rexp, gcn_cap):
        ei_exp  = snap['country', 'exports',     'product'].edge_index
        ei_rexp = snap['product', 'rev_exports', 'country'].edge_index
        ei_cap  = snap['product', 'capability',  'product'].edge_index
        xc_new  = sage_rexp((xp, xc), ei_rexp)
        xp_new  = sage_exp((xc, xp), ei_exp) + gcn_cap(xp, ei_cap, edge_weight=cap_ew)
        return xc_new, xp_new

    def forward(self, snap, cap_ew):
        xc = self.country_lin(snap['country'].x)
        xp = self.product_lin(snap['product'].x)
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_1, self.sage_rexp_1, self.gcn_cap_1)
        xc = F.dropout(xc.relu(), p=self.drop, training=self.training)
        xp = F.dropout(xp.relu(), p=self.drop, training=self.training)
        xc, xp = self._layer(xc, xp, snap, cap_ew,
                              self.sage_exp_2, self.sage_rexp_2, self.gcn_cap_2)
        return {'country': xc, 'product': xp}

# ── Shared link predictor ─────────────────────────────────────────────────────
class LinkPredictor(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden * 2, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, 1))
    def forward(self, zc, zp, ei):
        return self.mlp(torch.cat([zc[ei[0]], zp[ei[1]]], -1)).view(-1)

# ── Snapshot builder ──────────────────────────────────────────────────────────
def build_snap(year, c_x, p_x_dict=None, with_cap=False, use_v2=False, use_gat_pca=False):
    """
    Returns (HeteroData, ea_dict|ew_1d|None).
    - use_v2=True:      ea_dict for GATConv v2 (Methods 9 & 10)
    - use_gat_pca=True: ea_dict for GATConv PCA (Methods 13 & 14) — cosine as edge_attr [E,1]
    - with_cap + neither: ew_1d for GCNConv (Method 12) or None (Method 11 SAGEConv)
    - p_x_dict: override product features dict
    """
    if p_x_dict is None:
        p_x_dict = p_x_by_yr_v2 if use_v2 else p_x_by_yr
    d = HeteroData()
    d['country'].x = c_x[year]
    d['product'].x = p_x_dict[year]
    ei = edge_idx_by_yr[year].long()
    d['country', 'exports',     'product'].edge_index = ei
    d['product', 'rev_exports', 'country'].edge_index = ei.flip(0)
    ea = None
    if with_cap:
        d['product', 'capability', 'product'].edge_index = cap_ei
        if use_v2:
            d['product', 'capability', 'product'].edge_attr = cos_weights
            ea = {
                ('country', 'exports',     'product'): None,
                ('product', 'rev_exports', 'country'): None,
                ('product', 'capability',  'product'): cos_weights,
            }
        elif use_gat_pca:
            # GATConv PCA needs edge_attr shape [E, 1]
            d["product", "capability", "product"].edge_attr = cos_weights.unsqueeze(1)
            ea = {
                ("country", "exports",     "product"): None,
                ("product", "rev_exports", "country"): None,
                ("product", "capability",  "product"): cos_weights.unsqueeze(1),
            }
        else:
            ea = cos_weights_1d   # scalar weights for GCNConv
    return d, ea

def move_snap_to_dev(snap_pair, dev):
    s, ea = snap_pair
    s['country'].x = s['country'].x.to(dev)
    s['product'].x = s['product'].x.to(dev)
    for et in s.edge_types:
        s[et].edge_index = s[et].edge_index.to(device=dev, dtype=torch.long)
        if s[et].get('edge_attr') is not None:
            s[et].edge_attr = s[et].edge_attr.to(dev)
    if ea is not None:
        if isinstance(ea, dict):
            ea = {k: (v.to(dev) if v is not None else None) for k, v in ea.items()}
        else:
            ea = ea.to(dev)
    return s, ea

# ── Score builder: v1/v2 GNN on sampled test set ──────────────────────────────
@torch.no_grad()
def gnn_scores_sampled(ckpt_path, t, df_lbl, c_x, with_cap=False, use_v2=False):
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    if use_v2:
        enc  = BipartiteEncoderGAT(ckpt['c_in'], ckpt['p_in'],
                                   ckpt['hidden'], ckpt['heads'],
                                   ckpt['dropout'], ckpt['meta']).to(DEVICE)
        mdl  = TemporalGNNv2(enc, ckpt['hidden']).to(DEVICE)
    else:
        enc  = BipartiteEncoder(ckpt['c_in'], ckpt['hidden'], ckpt['meta']).to(DEVICE)
        mdl  = TemporalGNN(enc, ckpt['hidden']).to(DEVICE)
    pred = LinkPredictor(ckpt['hidden']).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    raw_snaps = [build_snap(y, c_x, with_cap=with_cap, use_v2=use_v2)
                 for y in range(t - 4, t + 1)]
    snaps = [move_snap_to_dev(sp, DEVICE) for sp in raw_snaps]

    ci_s = df_lbl['country'].map(c_map['to_idx'])
    pi_s = df_lbl['product'].map(p_map['to_idx'])
    ok   = ci_s.notna() & pi_s.notna()
    ei_t = torch.tensor([ci_s[ok].astype(int).values, pi_s[ok].astype(int).values],
                        dtype=torch.long).to(DEVICE)

    if use_v2:
        z = mdl(snaps)
    else:
        z = mdl([s for s, _ in snaps])
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    gnn_df = pd.DataFrame({'country': df_lbl.loc[ok, 'country'].values,
                           'product': df_lbl.loc[ok, 'product'].values,
                           'score':   raw})
    return df_lbl.merge(gnn_df[['country', 'product', 'score']],
                        on=['country', 'product'], how='left').fillna(0)['score'].values


# ── Score builder: PCA-based models (Variants A & B) ─────────────────────────
@torch.no_grad()
def gnn_scores_sampled_pca(ckpt_path, t, df_lbl, variant='A'):
    """
    Load and run a PCA-feature GNN checkpoint on the sampled test set.
    variant='A' → BipartiteEncoderSAGE_PCA + TemporalGNN_PCA (no edge weights)
    variant='B' → MixedBipartiteEncoder + TemporalGNN_PCA (cosine edge weights)
    """
    ckpt = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    hidden   = ckpt.get('hidden', 128)
    p_in     = ckpt.get('p_in', P_IN_PCA)
    pca_dim  = ckpt.get('pca_dim', PCA_DIM)

    # Build snapshots with capability edges; ew is cos_weights_1d for Variant B
    raw_snaps = [build_snap(y, c_x_11feat, p_x_dict=p_x_with_pca, with_cap=True)
                 for y in range(t - 4, t + 1)]
    snaps = [move_snap_to_dev(sp, DEVICE) for sp in raw_snaps]

    meta = snaps[0][0].metadata()

    if variant == 'A':
        enc = BipartiteEncoderSAGE_PCA(C_IN, p_in, hidden, drop=0.3, meta=meta).to(DEVICE)
    else:
        enc = MixedBipartiteEncoder(C_IN, p_in, hidden, drop=0.3).to(DEVICE)

    mdl  = TemporalGNN_PCA(enc, hidden).to(DEVICE)
    pred = LinkPredictor(hidden).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    ci_s = df_lbl['country'].map(c_map['to_idx'])
    pi_s = df_lbl['product'].map(p_map['to_idx'])
    ok   = ci_s.notna() & pi_s.notna()
    ei_t = torch.tensor([ci_s[ok].astype(int).values, pi_s[ok].astype(int).values],
                        dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    gnn_df = pd.DataFrame({'country': df_lbl.loc[ok, 'country'].values,
                           'product': df_lbl.loc[ok, 'product'].values,
                           'score':   raw})
    return df_lbl.merge(gnn_df[['country', 'product', 'score']],
                        on=['country', 'product'], how='left').fillna(0)['score'].values


# ── Variants C & D: GATConv + PCA features (Methods 13 & 14) ─────────────────
class _GATBlock_PCA(nn.Module):
    def __init__(self, hidden, heads, drop):
        super().__init__()
        self.gat1 = GATConv(hidden, hidden, heads=heads, concat=True,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.proj = nn.Linear(hidden * heads, hidden)
        self.gat2 = GATConv(hidden, hidden, heads=1, concat=False,
                             dropout=drop, edge_dim=1, add_self_loops=False, fill_value='mean')
        self.drop = drop
    def forward(self, x, edge_index, edge_attr=None):
        x = F.dropout(self.proj(self.gat1(x, edge_index, edge_attr=edge_attr).relu()),
                      p=self.drop, training=self.training)
        return self.gat2(x, edge_index, edge_attr=edge_attr)

class BipartiteEncoderGAT_PCA(nn.Module):
    def __init__(self, c_in, p_in, hidden, heads, drop, meta):
        super().__init__()
        self.country_lin = nn.Linear(c_in, hidden)
        self.product_lin = nn.Linear(p_in, hidden)
        self.gnn = to_hetero(_GATBlock_PCA(hidden, heads, drop), meta)
    def forward(self, x_dict, ei_dict, ea_dict=None):
        x_proj = {'country': self.country_lin(x_dict['country']),
                  'product': self.product_lin(x_dict['product'])}
        return self.gnn(x_proj, ei_dict, ea_dict) if ea_dict else self.gnn(x_proj, ei_dict)

class TemporalGNN_GAT_PCA(nn.Module):
    def __init__(self, enc, hidden):
        super().__init__()
        self.enc   = enc
        self.gru_c = nn.GRU(hidden, hidden)
        self.gru_p = nn.GRU(hidden, hidden)
    def forward(self, snaps):
        cs, ps = [], []
        for s, ea in snaps:
            z = self.enc(s.x_dict, s.edge_index_dict, ea)
            cs.append(z['country']); ps.append(z['product'])
        z_c, _ = self.gru_c(torch.stack(cs))
        z_p, _ = self.gru_p(torch.stack(ps))
        return {'country': z_c[-1], 'product': z_p[-1]}


# ── Score builder: Optimized GAT+PCA models (Variants C & D, Methods 13 & 14) ─
@torch.no_grad()
def gnn_scores_sampled_gat_pca(ckpt_path, t, df_lbl, variant='C'):
    ckpt   = torch.load(ckpt_path, weights_only=False, map_location=DEVICE)
    hidden = ckpt.get('hidden', 128)
    heads  = ckpt.get('heads', 2)
    drop   = ckpt.get('dropout', 0.3)
    p_in   = ckpt.get('p_in', P_IN_PCA)

    use_ew = (variant == 'D')
    raw_snaps = [build_snap(y, c_x_11feat, p_x_dict=p_x_with_pca,
                             with_cap=True, use_gat_pca=use_ew)
                 for y in range(t - 4, t + 1)]
    snaps = [move_snap_to_dev(sp, DEVICE) for sp in raw_snaps]

    meta = snaps[0][0].metadata()
    enc  = BipartiteEncoderGAT_PCA(C_IN, p_in, hidden, heads, drop, meta).to(DEVICE)
    mdl  = TemporalGNN_GAT_PCA(enc, hidden).to(DEVICE)
    pred = LinkPredictor(hidden).to(DEVICE)
    mdl.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['mdl_state'].items()})
    pred.load_state_dict({k: v.to(DEVICE) for k, v in ckpt['pred_state'].items()})
    mdl.eval(); pred.eval()

    ci_s = df_lbl['country'].map(c_map['to_idx'])
    pi_s = df_lbl['product'].map(p_map['to_idx'])
    ok   = ci_s.notna() & pi_s.notna()
    ei_t = torch.tensor([ci_s[ok].astype(int).values, pi_s[ok].astype(int).values],
                        dtype=torch.long).to(DEVICE)

    z   = mdl(snaps)
    raw = torch.sigmoid(pred(z['country'], z['product'], ei_t)).cpu().numpy()

    gnn_df = pd.DataFrame({'country': df_lbl.loc[ok, 'country'].values,
                           'product': df_lbl.loc[ok, 'product'].values,
                           'score':   raw})
    return df_lbl.merge(gnn_df[['country', 'product', 'score']],
                        on=['country', 'product'], how='left').fillna(0)['score'].values


# ── Checkpoint paths ──────────────────────────────────────────────────────────
CKPT_4F      = os.path.join(CKPT_DIR, 'gnn_4f.pt')
CKPT_11F     = os.path.join(CKPT_DIR, 'gnn_11f.pt')
CKPT_LLM     = os.path.join(CKPT_DIR, 'gnn_11f_llm.pt')
CKPT_UNOPT   = os.path.join(CKPT_DIR, 'gnn_llm_v2_unopt.pt')
CKPT_PCA_A   = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca.pt')
CKPT_PCA_B   = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_ew.pt')
CKPT_PCA_C   = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat.pt')
CKPT_PCA_D   = os.path.join(CKPT_DIR, 'gnn_11f_llm_pca_gat_ew.pt')
_v2_candidates = ['gnn_11f_llm_v2.pt', 'gnn_llm_v2.pt']
CKPT_V2 = None
for _name in _v2_candidates:
    _path = os.path.join(CKPT_DIR, _name)
    if os.path.exists(_path):
        CKPT_V2 = _path
        break

print('GNN helpers loaded.')
for name, path in [('4F', CKPT_4F), ('11F', CKPT_11F), ('LLM', CKPT_LLM),
                   ('v2 (Optuna)', CKPT_V2), ('v2 unopt', CKPT_UNOPT),
                   ('PCA-A (SAGE)', CKPT_PCA_A), ('PCA-B (GCN+EW)', CKPT_PCA_B),
                   ('PCA-C (GAT opt)', CKPT_PCA_C), ('PCA-D (GAT+EW opt)', CKPT_PCA_D)]:
    status = 'OK' if (path and os.path.exists(path)) else 'NOT FOUND'
    print(f'  GNN-{name}: {status}  ({path})')

GNN helpers loaded.
  GNN-4F: OK  (data\models\gnn\checkpoints\gnn_4f.pt)
  GNN-11F: OK  (data\models\gnn\checkpoints\gnn_11f.pt)
  GNN-LLM: OK  (data\models\gnn\checkpoints\gnn_11f_llm.pt)
  GNN-v2 (Optuna): OK  (data\models\gnn\checkpoints\gnn_11f_llm_v2.pt)


  GNN-v2 unopt: OK  (data\models\gnn\checkpoints\gnn_llm_v2_unopt.pt)
  GNN-PCA-A (SAGE): OK  (data\models\gnn\checkpoints\gnn_11f_llm_pca.pt)
  GNN-PCA-B (GCN+EW): OK  (data\models\gnn\checkpoints\gnn_11f_llm_pca_ew.pt)
  GNN-PCA-C (GAT opt): NOT FOUND  (data\models\gnn\checkpoints\gnn_11f_llm_pca_gat.pt)
  GNN-PCA-D (GAT+EW opt): OK  (data\models\gnn\checkpoints\gnn_11f_llm_pca_gat_ew.pt)


---
## Evaluation: t=2015 — Full Sampled Test Set

In [ ]:
T = 2015
df_t  = LABELS[T]
M_t, dens_mat, eci = build_year_structures(T)

ci_arr = df_t['country'].map(c_idx).values
pi_arr = df_t['product'].map(p_idx).values

print(f'=== t={T} | {len(df_t):,} pairs | {df_t["label"].mean()*100:.1f}% positive ===')

# ── Method 1: RCA Persistence ─────────────────────────────────────────────────
hist_yrs = [T - 2, T - 1, T]
rca_h = rca_df[rca_df['year'].isin(hist_yrs)][['country', 'product', 'year', 'rca']]
rca_h = rca_h.merge(df_t[['country', 'product']], on=['country', 'product'])
rca_w = rca_h.pivot_table(index=['country', 'product'], columns='year', values='rca', fill_value=0)
for yr in hist_yrs:
    if yr not in rca_w.columns: rca_w[yr] = 0
rca_w['score'] = (rca_w[hist_yrs] >= 1).mean(axis=1)
persist_sc = df_t.merge(rca_w[['score']], on=['country', 'product'], how='left').fillna(0)['score'].values
evaluate(T, 'RCA Persistence', persist_sc)

# ── Method 2: Density ─────────────────────────────────────────────────────────
evaluate(T, 'Density', dens_mat[ci_arr, pi_arr])

# ── Method 3: ECI ─────────────────────────────────────────────────────────────
evaluate(T, 'ECI', eci[ci_arr], skip_map10=True)

# ── Method 4: ECI + Density ───────────────────────────────────────────────────
evaluate(T, 'ECI + Density', minmax(eci[ci_arr]) + minmax(dens_mat[ci_arr, pi_arr]))

# ── Method 5: KNN (PCA-LLM embeddings, 32-dim) ───────────────────────────────
# Uses llm_pca_np [P, 32] — already fitted & L2-normalised in setup cell.
country_basket = {}
for c in df_t['country'].unique():
    ci = c_idx.get(c, -1)
    if ci < 0: continue
    exported = np.where(M_t[ci] == 1)[0]
    basket = llm_pca_np[exported].mean(axis=0) if len(exported) else np.zeros(PCA_DIM)
    norm = np.linalg.norm(basket)
    country_basket[c] = basket / (norm + 1e-9)
knn_sc = np.array([
    float(llm_pca_np[p_idx[p]] @ country_basket[c]) if c in country_basket and p in p_idx else 0.0
    for c, p in zip(df_t['country'].values, df_t['product'].values)
], dtype=np.float32)
evaluate(T, 'KNN (LLM embeddings)', knn_sc)

=== t=2015 | 127,531 pairs | 14.5% positive ===
  [2015] RCA Persistence                 PR-AUC=0.5198  NDCG@20=0.5013  Prec@20=0.4788  CWR=0.3359  P@1000=0.5970  mAP@10=0.3660
  [2015] Density                         PR-AUC=0.3487  NDCG@20=0.4809  Prec@20=0.4400  CWR=0.8574  P@1000=0.4860  mAP@10=0.3448
  [2015] ECI                             PR-AUC=0.1370  NDCG@20=0.1468  Prec@20=0.1374  CWR=0.4637  P@1000=0.0930  mAP@10=N/A
  [2015] ECI + Density                   PR-AUC=0.3487  NDCG@20=0.4809  Prec@20=0.4400  CWR=0.8574  P@1000=0.4860  mAP@10=0.3448
  [2015] KNN (LLM embeddings)            PR-AUC=0.2305  NDCG@20=0.2830  Prec@20=0.2644  CWR=0.6550  P@1000=0.4090  mAP@10=0.1644


{'PR-AUC': 0.2305,
 'AUROC': 0.6373,
 'NDCG@20': 0.283,
 'Prec@20': 0.2644,
 'CWR': 0.655,
 'Best F1': 0.2997,
 'P@1000': 0.409,
 'mAP@10': 0.1644}

In [22]:
T = 2015
df_t = LABELS[T]

# ── Method 6: GNN-4F ─────────────────────────────────────────────────────────
print(f'Loading GNN-4F ({CKPT_4F})...')
evaluate(T, 'GNN-4F', gnn_scores_sampled(CKPT_4F, T, df_t, c_x_4feat))

# ── Method 7: GNN-11F ────────────────────────────────────────────────────────
print(f'Loading GNN-11F...')
evaluate(T, 'GNN-11F (BACI+WDI)', gnn_scores_sampled(CKPT_11F, T, df_t, c_x_11feat))

# ── Method 8: GNN-11F+LLM ────────────────────────────────────────────────────
print(f'Loading GNN-11F+LLM...')
evaluate(T, 'GNN-11F+LLM', gnn_scores_sampled(CKPT_LLM, T, df_t, c_x_11feat, with_cap=True))

# ── Method 9: GNN-LLM v2 (Optuna) ────────────────────────────────────────────
if CKPT_V2:
    print(f'Loading GNN-LLM v2 ({CKPT_V2})...')
    evaluate(T, 'GNN-LLM v2 (GAT+Focal)', gnn_scores_sampled(CKPT_V2, T, df_t, c_x_11feat,
                                                               with_cap=True, use_v2=True))
else:
    print('GNN-LLM v2 checkpoint NOT FOUND — skipping Method 9')

# ── Method 10: GNN-LLM v2 Unopt (fixed hparams) ──────────────────────────────
if os.path.exists(CKPT_UNOPT):
    print(f'Loading GNN-LLM v2 Unopt ({CKPT_UNOPT})...')
    evaluate(T, 'GNN-LLM v2 Unopt', gnn_scores_sampled(CKPT_UNOPT, T, df_t, c_x_11feat,
                                                         with_cap=True, use_v2=True))
else:
    print('GNN-LLM v2 Unopt checkpoint NOT FOUND — skipping Method 10')

# ── Method 11: GNN-11F + LLM-PCA (SAGEConv, PCA features, no edge weights) ───
if os.path.exists(CKPT_PCA_A):
    print(f'Loading GNN-LLM PCA-A (SAGEConv+PCA)...')
    evaluate(T, 'GNN-LLM PCA-A (SAGE)', gnn_scores_sampled_pca(CKPT_PCA_A, T, df_t, variant='A'))
else:
    print(f'GNN-LLM PCA-A checkpoint NOT FOUND ({CKPT_PCA_A}) — run new_gnn_training_fixed.ipynb first')

# ── Method 12: GNN-11F + LLM-PCA + EdgeWeights (GCNConv with cosine weights) ─
if os.path.exists(CKPT_PCA_B):
    print(f'Loading GNN-LLM PCA-B (GCNConv+EW)...')
    evaluate(T, 'GNN-LLM PCA-B (GCN+EW)', gnn_scores_sampled_pca(CKPT_PCA_B, T, df_t, variant='B'))
else:
    print(f'GNN-LLM PCA-B checkpoint NOT FOUND ({CKPT_PCA_B}) — run new_gnn_training_fixed.ipynb first')

# ── Method 13: GNN-LLM PCA-C (GATConv + Focal + Optuna, no edge weights) ─────
if os.path.exists(CKPT_PCA_C):
    print(f'Loading GNN-LLM PCA-C (GAT+PCA, Optuna)...')
    evaluate(T, 'GNN-LLM PCA-C (GAT opt)', gnn_scores_sampled_gat_pca(CKPT_PCA_C, T, df_t, variant='C'))
else:
    print(f'GNN-LLM PCA-C checkpoint NOT FOUND ({CKPT_PCA_C}) — run new_gnn_training_fixed.ipynb first')

# ── Method 14: GNN-LLM PCA-D (GATConv + Focal + Optuna + cosine edge_attr) ───
if os.path.exists(CKPT_PCA_D):
    print(f'Loading GNN-LLM PCA-D (GAT+EW, Optuna)...')
    evaluate(T, 'GNN-LLM PCA-D (GAT+EW opt)', gnn_scores_sampled_gat_pca(CKPT_PCA_D, T, df_t, variant='D'))
else:
    print(f'GNN-LLM PCA-D checkpoint NOT FOUND ({CKPT_PCA_D}) — run new_gnn_training_fixed.ipynb first')

# ── XGBoost ──────────────────────────────────────────────────────────────
print(f'Running XGBoost for t={T}...')
evaluate(T, 'XGBoost', xgb_scores_sampled(T, df_t))


Loading GNN-4F (data\models\gnn\checkpoints\gnn_4f.pt)...
  [2015] GNN-4F                          PR-AUC=0.4123  NDCG@20=0.4369  Prec@20=0.3980  CWR=0.8838  P@1000=0.6200  mAP@10=0.2968
Loading GNN-11F...
  [2015] GNN-11F (BACI+WDI)              PR-AUC=0.4340  NDCG@20=0.4662  Prec@20=0.4257  CWR=0.8972  P@1000=0.6280  mAP@10=0.3306
Loading GNN-11F+LLM...
  [2015] GNN-11F+LLM                     PR-AUC=0.4408  NDCG@20=0.4848  Prec@20=0.4296  CWR=0.8989  P@1000=0.6460  mAP@10=0.3673
Loading GNN-LLM v2 (data\models\gnn\checkpoints\gnn_11f_llm_v2.pt)...
  [2015] GNN-LLM v2 (GAT+Focal)          PR-AUC=0.4108  NDCG@20=0.4414  Prec@20=0.3969  CWR=0.8871  P@1000=0.6560  mAP@10=0.3187
Loading GNN-LLM v2 Unopt (data\models\gnn\checkpoints\gnn_llm_v2_unopt.pt)...
  [2015] GNN-LLM v2 Unopt                PR-AUC=0.3190  NDCG@20=0.3155  Prec@20=0.2843  CWR=0.8501  P@1000=0.4090  mAP@10=0.1984
Loading GNN-LLM PCA-A (SAGEConv+PCA)...
  [2015] GNN-LLM PCA-A (SAGE)            PR-AUC=0.4500  NDCG@20=0.4

{'PR-AUC': 0.6576,
 'AUROC': 0.8992,
 'NDCG@20': 0.7004,
 'Prec@20': 0.6423,
 'CWR': 0.9376,
 'Best F1': 0.6319,
 'P@1000': 0.897,
 'mAP@10': 0.6128}

---
## Evaluation: t=2016 — Full Sampled Test Set

In [23]:
T = 2016
df_t  = LABELS[T]
M_t, dens_mat, eci = build_year_structures(T)

ci_arr = df_t['country'].map(c_idx).values
pi_arr = df_t['product'].map(p_idx).values

print(f'=== t={T} | {len(df_t):,} pairs | {df_t["label"].mean()*100:.1f}% positive ===')

# ── Method 1: RCA Persistence ─────────────────────────────────────────────────
hist_yrs = [T - 2, T - 1, T]
rca_h = rca_df[rca_df['year'].isin(hist_yrs)][['country', 'product', 'year', 'rca']]
rca_h = rca_h.merge(df_t[['country', 'product']], on=['country', 'product'])
rca_w = rca_h.pivot_table(index=['country', 'product'], columns='year', values='rca', fill_value=0)
for yr in hist_yrs:
    if yr not in rca_w.columns: rca_w[yr] = 0
rca_w['score'] = (rca_w[hist_yrs] >= 1).mean(axis=1)
persist_sc = df_t.merge(rca_w[['score']], on=['country', 'product'], how='left').fillna(0)['score'].values
evaluate(T, 'RCA Persistence', persist_sc)

# ── Method 2: Density ─────────────────────────────────────────────────────────
evaluate(T, 'Density', dens_mat[ci_arr, pi_arr])

# ── Method 3: ECI ─────────────────────────────────────────────────────────────
evaluate(T, 'ECI', eci[ci_arr], skip_map10=True)

# ── Method 4: ECI + Density ───────────────────────────────────────────────────
evaluate(T, 'ECI + Density', minmax(eci[ci_arr]) + minmax(dens_mat[ci_arr, pi_arr]))

# ── Method 5: KNN (PCA-LLM embeddings, 32-dim) ───────────────────────────────
# Uses llm_pca_np [P, 32] — already fitted & L2-normalised in setup cell.
country_basket = {}
for c in df_t['country'].unique():
    ci = c_idx.get(c, -1)
    if ci < 0: continue
    exported = np.where(M_t[ci] == 1)[0]
    basket = llm_pca_np[exported].mean(axis=0) if len(exported) else np.zeros(PCA_DIM)
    norm = np.linalg.norm(basket)
    country_basket[c] = basket / (norm + 1e-9)
knn_sc = np.array([
    float(llm_pca_np[p_idx[p]] @ country_basket[c]) if c in country_basket and p in p_idx else 0.0
    for c, p in zip(df_t['country'].values, df_t['product'].values)
], dtype=np.float32)
evaluate(T, 'KNN (LLM embeddings)', knn_sc)

=== t=2016 | 112,284 pairs | 16.7% positive ===
  [2016] RCA Persistence                 PR-AUC=0.5514  NDCG@20=0.5420  Prec@20=0.4896  CWR=0.3349  P@1000=0.0030  mAP@10=0.4651
  [2016] Density                         PR-AUC=0.4007  NDCG@20=0.4931  Prec@20=0.4573  CWR=0.8477  P@1000=0.5830  mAP@10=0.3633
  [2016] ECI                             PR-AUC=0.1563  NDCG@20=0.1656  Prec@20=0.8938  CWR=0.4322  P@1000=0.2970  mAP@10=N/A
  [2016] ECI + Density                   PR-AUC=0.4007  NDCG@20=0.4931  Prec@20=0.4573  CWR=0.8477  P@1000=0.5830  mAP@10=0.3633
  [2016] KNN (LLM embeddings)            PR-AUC=0.2609  NDCG@20=0.3151  Prec@20=0.2883  CWR=0.6504  P@1000=0.4600  mAP@10=0.1940


{'PR-AUC': 0.2609,
 'AUROC': 0.6369,
 'NDCG@20': 0.3151,
 'Prec@20': 0.2883,
 'CWR': 0.6504,
 'Best F1': 0.331,
 'P@1000': 0.46,
 'mAP@10': 0.194}

In [24]:
T = 2016
df_t = LABELS[T]

# ── Method 6: GNN-4F ─────────────────────────────────────────────────────────
print(f'Loading GNN-4F for t={T}...')
evaluate(T, 'GNN-4F', gnn_scores_sampled(CKPT_4F, T, df_t, c_x_4feat))

# ── Method 7: GNN-11F ────────────────────────────────────────────────────────
print(f'Loading GNN-11F for t={T}...')
evaluate(T, 'GNN-11F (BACI+WDI)', gnn_scores_sampled(CKPT_11F, T, df_t, c_x_11feat))

# ── Method 8: GNN-11F+LLM ────────────────────────────────────────────────────
print(f'Loading GNN-11F+LLM for t={T}...')
evaluate(T, 'GNN-11F+LLM', gnn_scores_sampled(CKPT_LLM, T, df_t, c_x_11feat, with_cap=True))

# ── Method 9: GNN-LLM v2 (Optuna) ────────────────────────────────────────────
if CKPT_V2:
    print(f'Loading GNN-LLM v2 for t={T}...')
    evaluate(T, 'GNN-LLM v2 (GAT+Focal)', gnn_scores_sampled(CKPT_V2, T, df_t, c_x_11feat,
                                                               with_cap=True, use_v2=True))
else:
    print('GNN-LLM v2 checkpoint NOT FOUND — skipping Method 9')

# ── Method 10: GNN-LLM v2 Unopt (fixed hparams) ──────────────────────────────
if os.path.exists(CKPT_UNOPT):
    print(f'Loading GNN-LLM v2 Unopt for t={T}...')
    evaluate(T, 'GNN-LLM v2 Unopt', gnn_scores_sampled(CKPT_UNOPT, T, df_t, c_x_11feat,
                                                         with_cap=True, use_v2=True))
else:
    print('GNN-LLM v2 Unopt checkpoint NOT FOUND — skipping Method 10')

# ── Method 11: GNN-11F + LLM-PCA (SAGEConv, PCA features, no edge weights) ───
if os.path.exists(CKPT_PCA_A):
    print(f'Loading GNN-LLM PCA-A for t={T}...')
    evaluate(T, 'GNN-LLM PCA-A (SAGE)', gnn_scores_sampled_pca(CKPT_PCA_A, T, df_t, variant='A'))
else:
    print(f'GNN-LLM PCA-A checkpoint NOT FOUND ({CKPT_PCA_A}) — run new_gnn_training_fixed.ipynb first')

# ── Method 12: GNN-11F + LLM-PCA + EdgeWeights (GCNConv with cosine weights) ─
if os.path.exists(CKPT_PCA_B):
    print(f'Loading GNN-LLM PCA-B for t={T}...')
    evaluate(T, 'GNN-LLM PCA-B (GCN+EW)', gnn_scores_sampled_pca(CKPT_PCA_B, T, df_t, variant='B'))
else:
    print(f'GNN-LLM PCA-B checkpoint NOT FOUND ({CKPT_PCA_B}) — run new_gnn_training_fixed.ipynb first')

# ── Method 13: GNN-LLM PCA-C (GATConv + Focal + Optuna, no edge weights) ─────
if os.path.exists(CKPT_PCA_C):
    print(f'Loading GNN-LLM PCA-C (GAT+PCA, Optuna) for t={T}...')
    evaluate(T, 'GNN-LLM PCA-C (GAT opt)', gnn_scores_sampled_gat_pca(CKPT_PCA_C, T, df_t, variant='C'))
else:
    print(f'GNN-LLM PCA-C checkpoint NOT FOUND ({CKPT_PCA_C}) — run new_gnn_training_fixed.ipynb first')

# ── Method 14: GNN-LLM PCA-D (GATConv + Focal + Optuna + cosine edge_attr) ───
if os.path.exists(CKPT_PCA_D):
    print(f'Loading GNN-LLM PCA-D (GAT+EW, Optuna) for t={T}...')
    evaluate(T, 'GNN-LLM PCA-D (GAT+EW opt)', gnn_scores_sampled_gat_pca(CKPT_PCA_D, T, df_t, variant='D'))
else:
    print(f'GNN-LLM PCA-D checkpoint NOT FOUND ({CKPT_PCA_D}) — run new_gnn_training_fixed.ipynb first')

# ── XGBoost ──────────────────────────────────────────────────────────────
print(f'Running XGBoost for t={T}...')
evaluate(T, 'XGBoost', xgb_scores_sampled(T, df_t))


Loading GNN-4F for t=2016...
  [2016] GNN-4F                          PR-AUC=0.4623  NDCG@20=0.4555  Prec@20=0.4150  CWR=0.8742  P@1000=0.6910  mAP@10=0.3332
Loading GNN-11F for t=2016...
  [2016] GNN-11F (BACI+WDI)              PR-AUC=0.4725  NDCG@20=0.4697  Prec@20=0.4361  CWR=0.8820  P@1000=0.6780  mAP@10=0.3342
Loading GNN-11F+LLM for t=2016...
  [2016] GNN-11F+LLM                     PR-AUC=0.4787  NDCG@20=0.4917  Prec@20=0.4458  CWR=0.8848  P@1000=0.7090  mAP@10=0.3686
Loading GNN-LLM v2 for t=2016...
  [2016] GNN-LLM v2 (GAT+Focal)          PR-AUC=0.4461  NDCG@20=0.4450  Prec@20=0.4066  CWR=0.8750  P@1000=0.6840  mAP@10=0.3172
Loading GNN-LLM v2 Unopt for t=2016...
  [2016] GNN-LLM v2 Unopt                PR-AUC=0.3641  NDCG@20=0.3360  Prec@20=0.3093  CWR=0.8449  P@1000=0.4760  mAP@10=0.2203
Loading GNN-LLM PCA-A for t=2016...
  [2016] GNN-LLM PCA-A (SAGE)            PR-AUC=0.4919  NDCG@20=0.5057  Prec@20=0.4584  CWR=0.8895  P@1000=0.7410  mAP@10=0.3786
Loading GNN-LLM PCA-B for

{'PR-AUC': 0.688,
 'AUROC': 0.8978,
 'NDCG@20': 0.7269,
 'Prec@20': 0.656,
 'CWR': 0.9285,
 'Best F1': 0.6567,
 'P@1000': 0.906,
 'mAP@10': 0.6506}

---
## Results Tables — Full Sampled Test Set

In [25]:
METHOD_ORDER = [
    'RCA Persistence', 'Density', 'ECI', 'ECI + Density',
    'KNN (LLM embeddings)', 'XGBoost', 'GNN-4F', 'GNN-11F (BACI+WDI)',
    'GNN-11F+LLM', 'GNN-LLM v2 (GAT+Focal)', 'GNN-LLM v2 Unopt',
    'GNN-LLM PCA-A (SAGE)', 'GNN-LLM PCA-B (GCN+EW)',
    'GNN-LLM PCA-C (GAT opt)', 'GNN-LLM PCA-D (GAT+EW opt)',
]
METRICS = ['PR-AUC', 'AUROC', 'NDCG@20', 'Prec@20', 'CWR', 'Best F1', 'P@1000', 'mAP@10']

def print_table(year, results):
    methods = [m for m in METHOD_ORDER if m in results]
    lbl_df  = LABELS[year]
    print(f'\nt={year} | {len(lbl_df):,} pairs | {lbl_df["label"].mean()*100:.1f}% positive rate')
    print('=' * 110)
    hdr = f'  {"Method":<32}' + ''.join(f'{m:>9}' for m in METRICS)
    print(hdr)
    print('-' * 110)
    rows = []
    for m in methods:
        r   = results[m]
        gnn = ' <' if 'GNN' in m else ''
        vals = []
        row_d = {'Method': m, 'Year': year}
        for met in METRICS:
            v = r.get(met, 'N/A')
            row_d[met] = v
            vals.append(f'{"N/A":>9}' if v == 'N/A' else f'{v:>9.4f}')
        print(f'  {m:<32}' + ''.join(vals) + gnn)
        rows.append(row_d)
    print('=' * 110)
    return rows

all_rows = []
for yr in [2015, 2016]:
    all_rows += print_table(yr, RESULTS[yr])

# Save combined CSV
os.makedirs('internal_benchmarking', exist_ok=True)
df_out = pd.DataFrame(all_rows).set_index(['Year', 'Method'])
df_out.to_csv('internal_benchmarking/full_sampled_results.csv')
print('\nSaved -> internal_benchmarking/full_sampled_results.csv')


t=2015 | 127,531 pairs | 14.5% positive rate
  Method                             PR-AUC    AUROC  NDCG@20  Prec@20      CWR  Best F1   P@1000   mAP@10
--------------------------------------------------------------------------------------------------------------
  RCA Persistence                    0.5198   0.6515   0.5013   0.4788   0.3359   0.4357   0.5970   0.3660
  Density                            0.3487   0.7792   0.4809   0.4400   0.8574   0.4205   0.4860   0.3448
  ECI                                0.1370   0.4821   0.1468   0.1374   0.4637   0.2597   0.0930      N/A
  ECI + Density                      0.3487   0.7792   0.4809   0.4400   0.8574   0.4205   0.4860   0.3448
  KNN (LLM embeddings)               0.2305   0.6373   0.2830   0.2644   0.6550   0.2997   0.4090   0.1644
  XGBoost                            0.6576   0.8992   0.7004   0.6423   0.9376   0.6319   0.8970   0.6128
  GNN-4F                             0.4123   0.8191   0.4369   0.3980   0.8838   0.4708   0.6

---
## RCA > 0.25 Filtered Evaluation

Filter the sampled test set to pairs where raw RCA[t] > 0.25 — country has genuine activity but not yet comparative advantage. Uses **cached scores** from the full evaluation above (no re-inference).

In [26]:
def build_rca025_filtered(t, df_lbl):
    """Filter df_lbl to pairs with RCA[t] > 0.25, add PCI weights, store orig_idx."""
    rca_t = rca_df[rca_df['year'] == t][['country', 'product', 'rca']]
    df    = df_lbl.copy()
    df['orig_idx'] = np.arange(len(df))   # positional index into full df
    df    = df.merge(rca_t, on=['country', 'product'], how='left')
    df['rca'] = df['rca'].fillna(0.0)
    df_f  = df[df['rca'] > RCA_THRESH].copy().reset_index(drop=True)
    # Recompute PCI weights on filtered subset
    df_f['pci'] = df_f['product'].map(pci_dict).fillna(pci_fill)
    min_pci_f   = df_f['pci'].min()
    df_f['w']   = df_f['pci'] - min_pci_f
    return df_f

def evaluate_filtered(t, name, full_scores):
    """Slice cached scores to the filtered subset and compute metrics."""
    df_f   = FILT_LABELS[t]
    scores = full_scores[df_f['orig_idx'].values]
    res    = compute_metrics(scores, df_f)
    RESULTS_FILT[t][name] = res
    tag = 'N/A' if res['mAP@10'] == 'N/A' else f'{res["mAP@10"]:.4f}'
    print(f'  [{t}|filt] {name:<30}  PR-AUC={res["PR-AUC"]:.4f}  '
          f'NDCG@20={res["NDCG@20"]:.4f}  CWR={res["CWR"]:.4f}  '
          f'P@1000={res["P@1000"]:.4f}  mAP@10={tag}')
    return res

# Build filtered subsets
FILT_LABELS = {}
for yr in [2015, 2016]:
    FILT_LABELS[yr] = build_rca025_filtered(yr, LABELS[yr])
    fl = FILT_LABELS[yr]
    print(f't={yr} RCA>0.25: {len(fl):,} pairs | {fl["label"].sum():,} pos | {fl["label"].mean()*100:.1f}% positive')

print()
# Evaluate all methods on filtered subsets using cached scores
for yr in [2015, 2016]:
    for name in METHOD_ORDER:
        if name not in SCORES[yr]:
            continue
        res = evaluate_filtered(yr, name, SCORES[yr][name])
        if name in INVALID_MAP10:
            RESULTS_FILT[yr][name]['mAP@10'] = 'N/A'

t=2015 RCA>0.25: 21,041 pairs | 11,648 pos | 55.4% positive
t=2016 RCA>0.25: 20,000 pairs | 11,961 pos | 59.8% positive

  [2015|filt] RCA Persistence                 PR-AUC=0.7396  NDCG@20=0.7071  CWR=0.4717  P@1000=0.7030  mAP@10=0.5387
  [2015|filt] Density                         PR-AUC=0.5808  NDCG@20=0.7375  CWR=0.5352  P@1000=0.6190  mAP@10=0.5951
  [2015|filt] ECI                             PR-AUC=0.5408  NDCG@20=0.5936  CWR=0.4875  P@1000=0.5230  mAP@10=0.3701
  [2015|filt] ECI + Density                   PR-AUC=0.5808  NDCG@20=0.7375  CWR=0.5352  P@1000=0.6190  mAP@10=0.5951
  [2015|filt] KNN (LLM embeddings)            PR-AUC=0.5856  NDCG@20=0.6280  CWR=0.5049  P@1000=0.6700  mAP@10=0.4450
  [2015|filt] XGBoost                         PR-AUC=0.7837  NDCG@20=0.8185  CWR=0.6628  P@1000=0.8970  mAP@10=0.7114
  [2015|filt] GNN-4F                          PR-AUC=0.6209  NDCG@20=0.7001  CWR=0.5333  P@1000=0.6630  mAP@10=0.5380
  [2015|filt] GNN-11F (BACI+WDI)              PR-AUC=

## Results Tables — RCA > 0.25 Filtered

In [27]:
filt_rows = []
for yr in [2015, 2016]:
    fl = FILT_LABELS[yr]
    methods = [m for m in METHOD_ORDER if m in RESULTS_FILT[yr]]
    print(f'\nt={yr} RCA>0.25 | {len(fl):,} pairs | {fl["label"].mean()*100:.1f}% positive rate')
    print('=' * 110)
    hdr = f'  {"Method":<32}' + ''.join(f'{m:>9}' for m in METRICS)
    print(hdr)
    print('-' * 110)
    for m in methods:
        r    = RESULTS_FILT[yr][m]
        gnn  = ' <' if 'GNN' in m else ''
        vals = []
        row_d = {'Method': m, 'Year': yr}
        for met in METRICS:
            v = r.get(met, 'N/A')
            row_d[met] = v
            vals.append(f'{'N/A':>9}' if v == 'N/A' else f'{v:>9.4f}')
        print(f'  {m:<32}' + ''.join(vals) + gnn)
        filt_rows.append(row_d)
    print('=' * 110)

df_filt_out = pd.DataFrame(filt_rows).set_index(['Year', 'Method'])
df_filt_out.to_csv('internal_benchmarking/filtered_rca025_results.csv')
print('\nSaved -> internal_benchmarking/filtered_rca025_results.csv')


t=2015 RCA>0.25 | 21,041 pairs | 55.4% positive rate
  Method                             PR-AUC    AUROC  NDCG@20  Prec@20      CWR  Best F1   P@1000   mAP@10
--------------------------------------------------------------------------------------------------------------
  RCA Persistence                    0.7396   0.6193   0.7071   0.5919   0.4717   0.7127   0.7030   0.5387
  Density                            0.5808   0.5422   0.7375   0.5944   0.5352   0.7157   0.6190   0.5951
  ECI                                0.5408   0.4814   0.5936   0.4924   0.4875   0.7127   0.5230      N/A
  ECI + Density                      0.5808   0.5422   0.7375   0.5944   0.5352   0.7157   0.6190   0.5951
  KNN (LLM embeddings)               0.5856   0.5294   0.6280   0.5166   0.5049   0.7127   0.6700   0.4450
  XGBoost                            0.7837   0.7662   0.8185   0.6659   0.6628   0.7601   0.8970   0.7114
  GNN-4F                             0.6209   0.5938   0.7001   0.5732   0.5333   0.71

---
## Cross-Year Consistency (t=2015 vs t=2016)

In [28]:
PRIMARY = ['PR-AUC', 'CWR', 'NDCG@20', 'P@1000']

print('=== Cross-year PR-AUC comparison (full sampled) ===')
print(f'  {"Method":<32} {"2015":>8} {"2016":>8} {"Î” (2016-2015)":>14}')
print('-' * 65)
for m in METHOD_ORDER:
    r15 = RESULTS[2015].get(m, {})
    r16 = RESULTS[2016].get(m, {})
    if not r15 or not r16:
        continue
    v15, v16 = r15['PR-AUC'], r16['PR-AUC']
    delta = v16 - v15
    print(f'  {m:<32} {v15:>8.4f} {v16:>8.4f} {delta:>+14.4f}')

print()
print('=== Cross-year PR-AUC comparison (RCA>0.25 filtered) ===')
print(f'  {"Method":<32} {"2015":>8} {"2016":>8} {"Î” (2016-2015)":>14}')
print('-' * 65)
for m in METHOD_ORDER:
    r15 = RESULTS_FILT[2015].get(m, {})
    r16 = RESULTS_FILT[2016].get(m, {})
    if not r15 or not r16:
        continue
    v15, v16 = r15['PR-AUC'], r16['PR-AUC']
    delta = v16 - v15
    print(f'  {m:<32} {v15:>8.4f} {v16:>8.4f} {delta:>+14.4f}')

=== Cross-year PR-AUC comparison (full sampled) ===
  Method                               2015     2016 Î” (2016-2015)
-----------------------------------------------------------------
  RCA Persistence                    0.5198   0.5514        +0.0316
  Density                            0.3487   0.4007        +0.0520
  ECI                                0.1370   0.1563        +0.0193
  ECI + Density                      0.3487   0.4007        +0.0520
  KNN (LLM embeddings)               0.2305   0.2609        +0.0304
  XGBoost                            0.6576   0.6880        +0.0304
  GNN-4F                             0.4123   0.4623        +0.0500
  GNN-11F (BACI+WDI)                 0.4340   0.4725        +0.0385
  GNN-11F+LLM                        0.4408   0.4787        +0.0379
  GNN-LLM v2 (GAT+Focal)             0.4108   0.4461        +0.0353
  GNN-LLM v2 Unopt                   0.3190   0.3641        +0.0451
  GNN-LLM PCA-A (SAGE)               0.4500   0.4919        +0.041

---
## Save All Results

In [29]:
os.makedirs('internal_benchmarking', exist_ok=True)

# Combined full-sampled CSV (both years)
df_full = pd.DataFrame(all_rows).set_index(['Year', 'Method'])
df_full.to_csv('internal_benchmarking/full_sampled_results.csv')

# Combined filtered CSV (both years)
df_filt = pd.DataFrame(filt_rows).set_index(['Year', 'Method'])
df_filt.to_csv('internal_benchmarking/filtered_rca025_results.csv')

# Per-year CSVs for convenience
for yr in [2015, 2016]:
    df_yr = pd.DataFrame([{'Method': m, **r} for m, r in RESULTS[yr].items()])
    df_yr.to_csv(f'internal_benchmarking/full_sampled_{yr}.csv', index=False)
    df_yf = pd.DataFrame([{'Method': m, **r} for m, r in RESULTS_FILT[yr].items()])
    df_yf.to_csv(f'internal_benchmarking/filtered_rca025_{yr}.csv', index=False)

print('Saved:')
for f in sorted(os.listdir('internal_benchmarking')):
    print(f'  internal_benchmarking/{f}')

Saved:
  internal_benchmarking/filtered_rca025_2015.csv
  internal_benchmarking/filtered_rca025_2016.csv
  internal_benchmarking/filtered_rca025_results.csv
  internal_benchmarking/full_sampled_2015.csv
  internal_benchmarking/full_sampled_2016.csv
  internal_benchmarking/full_sampled_results.csv


---
## Plots

Five figures saved to `internal_benchmarking/plots/`:

| # | File | Content |
|---|------|---------|
| 1 | `fig1_primary_metrics.png` | Grouped bar: PR-AUC / NDCG@20 / CWR / P@1000, 2015 vs 2016 |
| 2 | `fig2_heatmap_all_metrics.png` | Z-score heatmap, all 8 metrics x all methods |
| 3 | `fig3_cross_year_delta.png` | 2016 - 2015 delta bars per metric |
| 4 | `fig4_radar_top_methods.png` | Radar trade-off profile, top-5 representative methods |
| 5 | `fig5_filtered_vs_full.png` | PR-AUC full-sampled vs RCA>0.25 filtered |


In [ ]:
import subprocess, sys
from IPython.display import display, Image

script = os.path.join("ClaudeFiles", "plot_ib_results.py")
result = subprocess.run(
    [sys.executable, script],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
else:
    plot_dir = os.path.join("internal_benchmarking", "plots")
    for fname in sorted(os.listdir(plot_dir)):
        if fname.endswith(".png"):
            display(Image(filename=os.path.join(plot_dir, fname), width=900))
